# 🏥 HealthConnect Appointment Analysis

## Week 6: HealthConnect Integration, Advanced Development & Validation

**AnalystLab Africa — HealthConnect Experience Lab**

**Data Analytics Track**

Prepared by **Hudu Yusuf Ibrahim**

---

## Week 5 → Week 6 Transition

**Main Week 5 output:** A full analytics notebook covering data preparation, exploratory data analysis, five formally calculated KPIs, business insights, recommendations, and limitations — plus an initial Power BI dashboard with KPI cards, five comparison charts, a combined chart, and interactive slicers for appointment type and gender.

**Most important result:** Booking lead time showed the strongest observed relationship with No-Shows, with the rate rising from 27.8% (same-week bookings) to 56.9% (3+ weeks out). When combined with previous No-Show history, the highest observed No-Show rate reached 64.1%, compared with 21.8% for the lowest observed segment.

**Main issue/limitation discovered:** The Week 5 analysis is descriptive and correlational only. No statistical significance testing or predictive modelling was performed, so the strength of the observed patterns has not been formally validated. Additionally, the historical fields (`previous_appointments`, `previous_no_shows`) are synthetic and may not reflect coherent real-world patient histories, meaning conclusions involving these fields should be treated cautiously.

**Relevant track for Week 6:** Data Science. The Week 5 findings — particularly booking lead time and previous No-Show history as the two strongest observed signals — directly inform which features are worth prioritising in a No-Show prediction model.

**What Week 6 will improve, integrate, or validate:** This week will validate whether the two strongest observed relationships hold up under deeper segmentation, checking whether the patterns are broadly consistent or driven by specific subgroups. The analysis will also provide validated findings and explicit feature recommendations to the Data Science track as part of a documented cross-track integration activity.

---

In [1]:
import pandas as pd

df = pd.read_csv("HealthConnect_Appointment_Data.csv")
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['appointment_date'] = pd.to_datetime(df['appointment_date'])

df['lead_time_band'] = pd.cut(
    df['booking_lead_days'],
    bins=[-1, 7, 21, 60],
    labels=['Same week (0-7 days)', '2-3 weeks (8-21 days)', '3+ weeks (22-60 days)']
)

df['distance_band'] = pd.cut(
    df['distance_to_clinic_km'],
    bins=[0, 5, 15, 45],
    labels=['<5km', '5-15km', '15km+']
)

df['previous_no_show_group'] = df['previous_no_shows'].apply(
    lambda x: 'Previous No-Show' if x > 0 else 'No Previous No-Show'
)

df.shape

(5000, 21)

## Validating Booking Lead Time Across Appointment Types

In [2]:
pd.crosstab(
    [df['appointment_type'], df['lead_time_band']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

appointment_type         lead_time_band       
Diagnostic Test          Same week (0-7 days)     29.7
                         2-3 weeks (8-21 days)    32.4
                         3+ weeks (22-60 days)    59.0
Follow-up                Same week (0-7 days)     29.4
                         2-3 weeks (8-21 days)    40.4
                         3+ weeks (22-60 days)    59.9
General Consultation     Same week (0-7 days)     24.6
                         2-3 weeks (8-21 days)    37.9
                         3+ weeks (22-60 days)    54.4
Specialist Consultation  Same week (0-7 days)     30.6
                         2-3 weeks (8-21 days)    34.1
                         3+ weeks (22-60 days)    56.7
Name: No-Show, dtype: float64

### Observation

The relationship between booking lead time and No-Show rates is consistent across all four appointment types.

All appointment types show a substantial increase in observed No-Show rates for appointments booked 3+ weeks in advance compared with same-week bookings:

- **Diagnostic Test:** 29.7% → 59.0%
- **Follow-up:** 29.4% → 59.9%
- **General Consultation:** 24.6% → 54.4%
- **Specialist Consultation:** 30.6% → 56.7%

The increase ranges from **26.1 to 30.5 percentage points**, indicating that the Week 5 lead-time pattern is not concentrated in a single appointment type.

This provides additional support for treating booking lead time as an important characteristic for further investigation and potential inclusion in the Data Science team's No-Show prediction analysis.

However, this analysis remains descriptive and does not establish causation or statistical significance.

## Validating Previous No-Show History Across Appointment Types

In [3]:
pd.crosstab(
    [df['appointment_type'], df['previous_no_show_group']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

appointment_type         previous_no_show_group
Diagnostic Test          No Previous No-Show       44.7
                         Previous No-Show          56.5
Follow-up                No Previous No-Show       47.3
                         Previous No-Show          56.8
General Consultation     No Previous No-Show       40.4
                         Previous No-Show          55.3
Specialist Consultation  No Previous No-Show       43.9
                         Previous No-Show          52.8
Name: No-Show, dtype: float64

### Observation

The Week 5 finding that previous No-Show history is associated with higher observed No-Show rates was tested across all four appointment types.

| Appointment Type | No Previous No-Show | Previous No-Show | Gap |
|---|---:|---:|---:|
| Diagnostic Test | 44.7% | 56.5% | 11.8 pts |
| Follow-up | 47.3% | 56.8% | 9.5 pts |
| General Consultation | 40.4% | 55.3% | 14.9 pts |
| Specialist Consultation | 43.9% | 52.8% | 8.9 pts |

The pattern is consistent across all four appointment types: appointments involving patients with a previous No-Show history have a higher observed No-Show rate than those without previous No-Show history.

The observed gaps range from **8.9 to 14.9 percentage points**. General Consultation shows the largest gap (**14.9 pts**), while Specialist Consultation shows the smallest (**8.9 pts**).

The direction of the relationship is therefore consistent across appointment types, providing additional evidence that previous No-Show history is an important characteristic to investigate further in the Week 6 analysis and for potential consideration as a feature by the Data Science track.

However, the analysis remains descriptive. Statistical significance has not been tested, and the historical variables are synthetic, so these results should not be interpreted as proof of causation or as a validated predictive feature.

## Validating the Highest-Risk Segment (3+ Weeks + Previous No-Show) Across Appointment Types

In [4]:
pd.crosstab(
    [df['appointment_type'], df['lead_time_band'], df['previous_no_show_group']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

appointment_type         lead_time_band         previous_no_show_group
Diagnostic Test          Same week (0-7 days)   No Previous No-Show       16.7
                                                Previous No-Show          46.4
                         2-3 weeks (8-21 days)  No Previous No-Show       35.9
                                                Previous No-Show          27.6
                         3+ weeks (22-60 days)  No Previous No-Show       52.2
                                                Previous No-Show          68.0
Follow-up                Same week (0-7 days)   No Previous No-Show       23.7
                                                Previous No-Show          36.7
                         2-3 weeks (8-21 days)  No Previous No-Show       36.3
                                                Previous No-Show          46.7
                         3+ weeks (22-60 days)  No Previous No-Show       56.4
                                                Previous No-

### Observation

The Week 5 finding that the combination of **3+ week booking lead time and previous No-Show history** produces the highest observed No-Show rate (64.1% overall) was further examined across all four appointment types.

| Appointment Type | No-Show Rate (3+ weeks, Previous No-Show) |
|---|---:|
| Diagnostic Test | 68.0% |
| Follow-up | 64.8% |
| General Consultation | 63.3% |
| Specialist Consultation | 61.9% |

The **3+ week lead time + previous No-Show** segment has a high observed No-Show rate across all four appointment types, ranging from **61.9% to 68.0%**.

The spread between the highest and lowest values is **6.1 percentage points**, indicating that the high observed No-Show rate is broadly consistent across appointment types rather than being concentrated in only one category.

This provides additional evidence that the combination of **longer booking lead time and previous No-Show history** is an important pattern for further investigation and a relevant candidate for consideration by the Data Science track.

However, this remains a descriptive validation. Statistical significance and predictive performance have not been tested, so the combination should not yet be described as a confirmed risk indicator or causal factor.

### Additional Observation

Some three-way segments show less consistent patterns. For example, within Diagnostic Test appointments booked 2–3 weeks in advance, the observed No-Show rate is **27.6%** for patients with previous No-Shows compared with **35.9%** for those without previous No-Shows.

This reversal does not follow the broader pattern observed in the other segments. It may reflect subgroup size or sampling variation, so it should be investigated further rather than treated as evidence of a genuine reversal.

The main Week 6 finding remains that the **3+ week + previous No-Show** segment consistently has a high observed No-Show rate across all four appointment types.

## Sample Size Check for Segment Validation

In [5]:
pd.crosstab(
    [df['appointment_type'], df['lead_time_band'], df['previous_no_show_group']],
    df['appointment_outcome']
)

appointment_outcome                                                   Attended  \
appointment_type        lead_time_band        previous_no_show_group             
Diagnostic Test         Same week (0-7 days)  No Previous No-Show           29   
                                              Previous No-Show              15   
                        2-3 weeks (8-21 days) No Previous No-Show           47   
                                              Previous No-Show              38   
                        3+ weeks (22-60 days) No Previous No-Show           94   
                                              Previous No-Show              49   
Follow-up               Same week (0-7 days)  No Previous No-Show           80   
                                              Previous No-Show              48   
                        2-3 weeks (8-21 days) No Previous No-Show          113   
                                              Previous No-Show              63   
                        3+ weeks (22-60 days) No Previous No-Show          193   
                                              Previous No-Show             119   
General Consultation    Same week (0-7 days)  No Previous No-Show          113   
                                              Previous No-Show              66   
                        2-3 weeks (8-21 days) No Previous No-Show          197   
                                              Previous No-Show             104   
                        3+ weeks (22-60 days) No Previous No-Show          340   
                                              Previous No-Show             187   
Specialist Consultation Same week (0-7 days)  No Previous No-Show           52   
                                              Previous No-Show              23   
                        2-3 weeks (8-21 days) No Previous No-Show           86   
                                              Previous No-Show              52   
                        3+ weeks (22-60 days) No Previous No-Show          130   
                                              Previous No-Show              76   

appointment_outcome                                                   Cancelled  \
appointment_type        lead_time_band        previous_no_show_group              
Diagnostic Test         Same week (0-7 days)  No Previous No-Show             1   
                                              Previous No-Show                0   
                        2-3 weeks (8-21 days) No Previous No-Show             3   
                                              Previous No-Show                4   
                        3+ weeks (22-60 days) No Previous No-Show            13   
                                              Previous No-Show                5   
Follow-up               Same week (0-7 days)  No Previous No-Show             7   
                                              Previous No-Show                9   
                        2-3 weeks (8-21 days) No Previous No-Show             8   
                                              Previous No-Show                2   
                        3+ weeks (22-60 days) No Previous No-Show            37   
                                              Previous No-Show               14   
General Consultation    Same week (0-7 days)  No Previous No-Show             5   
                                              Previous No-Show                3   
                        2-3 weeks (8-21 days) No Previous No-Show            16   
                                              Previous No-Show               12   
                        3+ weeks (22-60 days) No Previous No-Show            51   
                                              Previous No-Show               19   
Specialist Consultation Same week (0-7 days)  No Previous No-Show             4   
                                              Previous No-Show                7   
                        2-3 weeks (8-21 d

### Observation

To assess the reliability of the segment-level findings, the underlying record counts were reviewed for each combination.

### Highest Observed Segment: 3+ Weeks + Previous No-Show

The four appointment types have substantial numbers of records in the **3+ weeks + Previous No-Show** segment:

| Appointment Type | Total Records | No-Show Rate |
|---|---:|---:|
| Diagnostic Test | 169 | 68.0% |
| Follow-up | 378 | 64.8% |
| General Consultation | 561 | 63.3% |
| Specialist Consultation | 226 | 61.9% |

These sample sizes provide a stronger basis for interpreting the observed 61.9%–68.0% range than would be possible with very small subgroups.

The consistency across four appointment types, together with the underlying sample sizes, provides additional evidence that the high observed No-Show rate for the **3+ weeks + Previous No-Show** segment is not concentrated in a single appointment type.

However, sample size alone does not establish statistical significance or predictive reliability. Further statistical testing would be required to formally assess the strength of this relationship.

### Anomalous Segment: Diagnostic Test, 2–3 Weeks

The earlier analysis identified an unexpected pattern within Diagnostic Test appointments booked 2–3 weeks in advance:

- No Previous No-Show: **35.9%** (28 No-Shows out of 78 records)
- Previous No-Show: **27.6%** (16 No-Shows out of 58 records)

The Previous No-Show subgroup contains **58 records**, which is considerably smaller than the larger segments examined above.

The smaller subgroup size may contribute to greater variation in the observed rate. Therefore, this result should be treated cautiously and should not be interpreted as evidence that previous No-Show history reduces No-Shows for Diagnostic Tests.

This demonstrates the importance of reviewing both **rates and underlying record counts** when interpreting highly segmented results.

---

## Statistical Significance Testing

The Week 5 analysis was descriptive and correlational, with no formal statistical testing performed. This was identified as a limitation. Week 6 addresses this limitation by applying statistical tests to the two strongest findings from Week 5.

### Booking Lead Time vs. Appointment Outcome

A **Chi-Square test of independence** was used to determine whether booking lead time and appointment outcome are statistically associated.

In [6]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['lead_time_band'], df['appointment_outcome'])
chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {p_value:.10f}")
print(f"Degrees of freedom: {dof}")

Chi-square statistic: 280.09
p-value: 0.0000000000
Degrees of freedom: 4


### Observation

The Chi-Square test produced a **chi-square statistic of 280.09**, with **4 degrees of freedom** and a **p-value below 0.0001**.

Since the p-value is well below the commonly used 0.05 significance threshold, there is **strong statistical evidence of an association between booking lead time and appointment outcome** in this dataset.

This provides formal statistical support for the Week 5 observation that appointment outcomes differ across booking lead-time groups. The result strengthens the evidence that the observed relationship is unlikely to be explained by random variation alone.

However, statistical significance does not establish causation or indicate the practical strength of the relationship. Further analysis using an effect-size measure is needed to assess the strength of the association.

### Previous No-Show History vs. Appointment Outcome

A **Chi-Square test of independence** was used to determine whether previous No-Show history and appointment outcome are statistically associated.

In [7]:
contingency_table2 = pd.crosstab(df['previous_no_show_group'], df['appointment_outcome'])
chi2_2, p_value_2, dof_2, expected_2 = chi2_contingency(contingency_table2)

print(f"Chi-square statistic: {chi2_2:.2f}")
print(f"p-value: {p_value_2:.10f}")
print(f"Degrees of freedom: {dof_2}")

Chi-square statistic: 69.86
p-value: 0.0000000000
Degrees of freedom: 2


### Observation

The Chi-Square test produced a **chi-square statistic of 69.86**, with **2 degrees of freedom** and a **p-value below 0.0001**.

Since the p-value is well below the commonly used 0.05 significance threshold, there is **strong statistical evidence of an association between previous No-Show history and appointment outcome** in this dataset.

This provides formal statistical support for the Week 5 observation that patients with and without a previous No-Show history have different observed appointment outcomes.

However, statistical significance does not establish causation or indicate the practical strength of the relationship. An effect-size measure is needed to compare the strength of this association with booking lead time.

## Statistical Validation Summary

Both of the two strongest Week 5 findings have now been formally tested:

| Factor | Chi-square | df | p-value | Statistically Significant? |
|---|---:|---:|---|---|
| Booking Lead Time | 280.09 | 4 | < 0.0001 | Yes |
| Previous No-Show History | 69.86 | 2 | < 0.0001 | Yes |

Both tests provide strong statistical evidence that the respective variables are associated with appointment outcomes in this dataset.

This addresses the Week 5 limitation regarding the absence of formal statistical testing. However, statistical significance does not imply causation, and the chi-square statistics should not be used alone to compare the strength of the two relationships.

The next step is to calculate **Cramér's V** to measure the practical strength of each association and provide a more appropriate comparison between the two factors.

## Measuring Association Strength with Cramér's V

The Chi-Square tests established that both booking lead time and previous No-Show history are statistically associated with appointment outcome.

However, statistical significance does not indicate how strong these associations are. To assess their practical strength and allow a more appropriate comparison between the two factors, **Cramér's V** will be calculated.

Cramér's V ranges from **0 to 1**, where values closer to 0 indicate a weaker association and values closer to 1 indicate a stronger association.

In [8]:
import numpy as np

def cramers_v(chi2_stat, contingency_table):
    n = contingency_table.sum().sum()
    min_dim = min(contingency_table.shape) - 1
    return np.sqrt(chi2_stat / (n * min_dim))

# Booking Lead Time
v_lead_time = cramers_v(chi2, contingency_table)
print(f"Cramér's V (Lead Time Band): {v_lead_time:.3f}")

# Previous No-Show History
v_previous_no_show = cramers_v(chi2_2, contingency_table2)
print(f"Cramér's V (Previous No-Show Group): {v_previous_no_show:.3f}")

Cramér's V (Lead Time Band): 0.167
Cramér's V (Previous No-Show Group): 0.118


### Observation

The Cramér’s V results show that both variables have relatively small associations with appointment outcome:

- **Booking Lead Time:** Cramér’s V = **0.167**
- **Previous No-Show History:** Cramér’s V = **0.118**

Booking lead time has the stronger association of the two, although the difference in association strength is modest.

This adds an important layer to the Chi-square results. While the Chi-square tests provided statistical evidence of an association, Cramér’s V shows that the overall strength of these associations is relatively small.

The results suggest that **booking lead time and previous No-Show history are relevant characteristics, but neither variable should be considered sufficient on its own to explain No-Show behaviour**.

This supports the Week 6 objective of moving beyond individual factors and considering multiple characteristics together. The findings are also relevant to the Data Science track, where these variables can be considered alongside other features when developing and evaluating a No-Show prediction model.

However, Cramér’s V measures association strength rather than predictive performance. Therefore, these results should not be interpreted as evidence that either variable is a strong predictor without further predictive modelling and validation.

## Validating Lead Time Across Reminder Status

The Week 5 analysis showed that booking lead time had the strongest observed relationship with No-Show rates.

To further validate this finding, the relationship between booking lead time and No-Show rate will be examined separately for appointments where a reminder was sent and where no reminder was sent.

This helps determine whether the lead-time pattern remains consistent across different reminder statuses rather than being concentrated in only one group.

In [9]:
pd.crosstab(
    [df['reminder_sent'], df['lead_time_band']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

reminder_sent  lead_time_band       
No             Same week (0-7 days)     29.9
               2-3 weeks (8-21 days)    40.8
               3+ weeks (22-60 days)    59.6
Yes            Same week (0-7 days)     27.1
               2-3 weeks (8-21 days)    35.8
               3+ weeks (22-60 days)    55.9
Name: No-Show, dtype: float64

### Observation

The increase in No-Show rates across booking lead-time bands is consistent for both reminder groups.

| Reminder Status | Same Week | 2–3 Weeks | 3+ Weeks | Spread |
|---|---:|---:|---:|---:|
| No reminder | 29.9% | 40.8% | 59.6% | 29.7 pts |
| Reminder sent | 27.1% | 35.8% | 55.9% | 28.8 pts |

For appointments with **no reminder**, the No-Show rate increases from **29.9%** for same-week bookings to **59.6%** for bookings made 3+ weeks in advance.

For appointments where a **reminder was sent**, the rate increases from **27.1%** to **55.9%** across the same lead-time bands.

The overall increase is therefore very similar in both groups (**29.7 vs. 28.8 percentage points**). This provides additional evidence that the Week 5 lead-time pattern is consistent across reminder status rather than appearing only within one reminder group.

This strengthens the case for further investigating booking lead time as an important characteristic for the No-Show analysis. However, this analysis does not establish independence, causation, or predictive performance.

## Validating the Highest-Observed Segment Across Reminder Status

The Week 5 analysis identified **3+ weeks of booking lead time combined with previous No-Show history** as the highest-observed No-Show segment, with an overall No-Show rate of **64.1%**.

To further validate this finding, the segment was examined separately for appointments where a reminder was sent and where no reminder was sent.

In [10]:
pd.crosstab(
    [df['reminder_sent'], df['lead_time_band'], df['previous_no_show_group']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

reminder_sent  lead_time_band         previous_no_show_group
No             Same week (0-7 days)   No Previous No-Show       22.7
                                      Previous No-Show          40.0
               2-3 weeks (8-21 days)  No Previous No-Show       35.4
                                      Previous No-Show          48.6
               3+ weeks (22-60 days)  No Previous No-Show       53.8
                                      Previous No-Show          67.4
Yes            Same week (0-7 days)   No Previous No-Show       21.5
                                      Previous No-Show          34.8
               2-3 weeks (8-21 days)  No Previous No-Show       33.6
                                      Previous No-Show          39.2
               3+ weeks (22-60 days)  No Previous No-Show       50.9
                                      Previous No-Show          62.8
Name: No-Show, dtype: float64

In [11]:
pd.crosstab(
    [df['reminder_sent'], df['lead_time_band'], df['previous_no_show_group']],
    df['appointment_outcome']
)

appointment_outcome                                         Attended  \
reminder_sent lead_time_band        previous_no_show_group             
No            Same week (0-7 days)  No Previous No-Show           72   
                                    Previous No-Show              38   
              2-3 weeks (8-21 days) No Previous No-Show          112   
                                    Previous No-Show              64   
              3+ weeks (22-60 days) No Previous No-Show          189   
                                    Previous No-Show             108   
Yes           Same week (0-7 days)  No Previous No-Show          202   
                                    Previous No-Show             114   
              2-3 weeks (8-21 days) No Previous No-Show          331   
                                    Previous No-Show             193   
              3+ weeks (22-60 days) No Previous No-Show          568   
                                    Previous No-Show             323   

appointment_outcome                                         Cancelled  No-Show  
reminder_sent lead_time_band        previous_no_show_group                      
No            Same week (0-7 days)  No Previous No-Show             3       22  
                                    Previous No-Show                4       28  
              2-3 weeks (8-21 days) No Previous No-Show            14       69  
                                    Previous No-Show                7       67  
              3+ weeks (22-60 days) No Previous No-Show            41      268  
                                    Previous No-Show               12      248  
Yes           Same week (0-7 days)  No Previous No-Show            14       59  
                                    Previous No-Show               15       69  
              2-3 weeks (8-21 days) No Previous No-Show            22      179  
                                    Previous No-Show               13      133  
              3+ weeks (22-60 days) No Previous No-Show            82      674  
                                    Previous No-Show               36      607

### Observation

The combined **3+ weeks + Previous No-Show** segment remains associated with a high observed No-Show rate across both reminder groups.

The No-Show rate is **67.4%** among appointments without a reminder and **62.8%** among appointments where a reminder was sent. The difference between the two groups is **4.6 percentage points**.

Both groups have substantial sample sizes, with **368 records** in the no-reminder group and **966 records** in the reminder-sent group, providing a reasonable basis for comparing the observed rates.

The consistency of the high No-Show rates across reminder status provides additional evidence that the combined lead-time and previous No-Show pattern is not concentrated within only one reminder group.

This strengthens the case for prioritising the **3+ weeks + Previous No-Show** segment for further investigation and potential consideration in the Data Science team's No-Show modelling work.

However, the analysis remains descriptive. The results do not establish that reminders are ineffective or that another intervention would necessarily reduce No-Shows. Further statistical and predictive analysis would be required to evaluate the effectiveness of different interventions.

## Validating Booking Lead Time Across Distance Bands

Week 5 identified **booking lead time** as the strongest observed relationship with appointment No-Show rates.

To further validate this finding, the relationship between booking lead time and No-Show rates will be examined across different **distance-to-clinic bands**.

This analysis will assess whether the increase in No-Show rates for appointments booked further in advance remains consistent for patients living different distances from the clinic.

The results will help determine whether the Week 5 lead-time pattern is broadly consistent or concentrated within particular distance groups.

In [12]:
pd.crosstab(
    [df['distance_band'], df['lead_time_band']],
    df['appointment_outcome'],
    normalize='index'
)['No-Show'].mul(100).round(1)

distance_band  lead_time_band       
<5km           Same week (0-7 days)     24.1
               2-3 weeks (8-21 days)    29.5
               3+ weeks (22-60 days)    57.8
5-15km         Same week (0-7 days)     26.1
               2-3 weeks (8-21 days)    37.8
               3+ weeks (22-60 days)    54.9
15km+          Same week (0-7 days)     35.9
               2-3 weeks (8-21 days)    44.7
               3+ weeks (22-60 days)    61.8
Name: No-Show, dtype: float64

In [13]:
pd.crosstab(
    [df['distance_band'], df['lead_time_band']],
    df['appointment_outcome']
)

appointment_outcome                  Attended  Cancelled  No-Show
distance_band lead_time_band                                     
<5km          Same week (0-7 days)        112          8       38
              2-3 weeks (8-21 days)       180         14       81
              3+ weeks (22-60 days)       274         31      418
5-15km        Same week (0-7 days)        232         20       89
              2-3 weeks (8-21 days)       394         31      258
              3+ weeks (22-60 days)       702        109      989
15km+         Same week (0-7 days)         74          8       46
              2-3 weeks (8-21 days)       115         10      101
              3+ weeks (22-60 days)       192         28      356

### Observation

The increase in No-Show rates across booking lead-time bands is consistent across all three distance groups.

For patients living **less than 5km** from the clinic, the No-Show rate increases from **24.1%** for same-week bookings to **57.8%** for bookings made 3+ weeks in advance.

For patients living **5–15km** away, the rate increases from **26.1%** to **54.9%**, while for patients living **15km or more** away, it increases from **35.9%** to **61.8%**.

The increase ranges from **25.9 to 33.7 percentage points**, showing that the Week 5 lead-time pattern is observed across all three distance groups rather than being concentrated in one group.

The **15km+** group has the highest No-Show rate at every lead-time band, which is consistent with the Week 5 finding that greater distance is associated with a higher observed No-Show rate.

The results provide additional evidence that booking lead time is an important characteristic to investigate further. However, this analysis remains descriptive and does not establish causation, independence, or predictive performance.

## Statistical Validation of the Combined Segment (Lead Time × Previous No-Show History)

The Week 5 analysis identified the combination of **booking lead time** and **previous No-Show history** as an important observed pattern.

The individual factors were previously tested using Chi-Square and Cramér's V. To provide additional statistical validation of the combined segmentation, the two variables were combined into a six-category variable and tested against appointment outcome.

This analysis assesses whether the combined categories are statistically associated with appointment outcomes and compares the strength of this association with the individual factors.

In [14]:
df['combined_segment'] = df['lead_time_band'].astype(str) + " | " + df['previous_no_show_group']

contingency_table3 = pd.crosstab(df['combined_segment'], df['appointment_outcome'])
chi2_3, p_value_3, dof_3, expected_3 = chi2_contingency(contingency_table3)

v_combined = cramers_v(chi2_3, contingency_table3)

print(f"Chi-square statistic: {chi2_3:.2f}")
print(f"p-value: {p_value_3:.10f}")
print(f"Cramér's V (Combined Segment): {v_combined:.3f}")

Chi-square statistic: 358.52
p-value: 0.0000000000
Cramér's V (Combined Segment): 0.189


### Observation

The combined six-category variable shows a statistically significant association with appointment outcome:

- **Chi-square = 358.52**
- **p < 0.0001**
- **Cramér's V = 0.189**

The Cramér's V for the combined segment is higher than the values obtained for the individual factors:

| Factor | Chi-square | Cramér's V |
|---|---:|---:|
| Booking Lead Time | 280.09 | 0.167 |
| Previous No-Show History | 69.86 | 0.118 |
| **Combined Segment** | **358.52** | **0.189** |

Among the three tested associations, the **combined segment has the highest Cramér's V**, indicating that the six-category combination has the strongest observed association with appointment outcome in this analysis.

This provides additional statistical support for the Week 5 finding that **booking lead time and previous No-Show history are useful when considered together**, rather than examining each characteristic separately.

The result is particularly relevant to the Data Science track because the combined segmentation can be considered as a candidate feature or interaction to investigate during predictive modelling.

However, this test does not establish that the two variables interact in a causal or predictive sense. Cramér's V measures association, not predictive performance. Further modelling and validation by the Data Science track will therefore be needed to determine whether the combined relationship improves prediction.

---

## KPI Refinement (Week 6)

Following the deeper analysis and statistical validation performed in Week 6, the Week 5 KPIs were reviewed to determine whether their definitions, priority, or dashboard presence should be refined.

| KPI | Week 5 Status | Week 6 Status | Rationale |
|---|---|---|---|
| Overall No-Show Rate | Baseline KPI | **Unchanged** | Remains the primary baseline measure for monitoring overall appointment No-Show levels. |
| No-Show Rate by Booking Lead Time | KPI 3 | **Confirmed, elevated priority** | Statistically associated with appointment outcome (χ²=280.09, Cramér's V=0.167). The observed increase in No-Show rate remained consistent across appointment types, distance bands, and reminder-status groups. |
| No-Show Rate by Previous No-Show History | KPI 4 | **Confirmed, second priority** | Statistically associated with appointment outcome (χ²=69.86, Cramér's V=0.118). The higher observed No-Show rate among patients with previous No-Shows remained consistent across appointment types. |
| **Combined Lead Time × Previous No-Show Segment** | Reported as "Insight #6" | **Elevated to priority analysis** | The combined six-category segment showed the highest Cramér's V among the tested factors (χ²=358.52, Cramér's V=0.189). The 3+ weeks + Previous No-Show segment also had the highest observed No-Show rate at 64.1%. |
| No-Show Rate by Distance to Clinic | KPI 5 | **Retained as supporting KPI** | The observed No-Show rate increased across distance bands, while the lead-time pattern remained visible within each distance group. Distance is therefore retained as a supporting factor rather than a primary focus. |
| No-Show Rate by Reminder Status/Channel | KPI 2 | **Retained as supporting analysis** | Differences between reminder groups and channels were relatively small. The lead-time pattern also remained consistent across reminder-status groups, so reminder variables remain useful for monitoring and further investigation. |
| No-Show Rate by Appointment Type / Gender | Tested in Week 5 (Insight #8) | **Retained as dashboard filters only** | Observed differences were relatively small compared with the main findings. These variables remain useful for interactive segmentation rather than being promoted to core KPIs. |

### Summary of Changes

The Week 5 KPI structure remains broadly appropriate, but Week 6 provides stronger evidence for prioritising the main findings.

**Booking lead time** remains the highest-priority individual factor, followed by **previous No-Show history**. The combined **Lead Time × Previous No-Show** analysis has also been elevated because it produced the highest Cramér's V among the tested factors and identified a high-observed-No-Show segment of **64.1%**.

Distance and reminder variables remain useful supporting analyses, while appointment type and gender are retained primarily as dashboard filters.

The Week 6 refinement therefore focuses on **prioritisation and validation rather than replacing the original KPI framework**.

---

## Week 6 Update: Statistically Validated Insights

The Week 5 insights above were descriptive and correlational. Week 6 added formal statistical testing using Chi-square tests of independence and Cramér's V effect sizes to assess whether the key observed relationships were statistically supported and to quantify their association strength.

**9. Booking Lead Time and Previous No-Show History Show Statistically Significant Associations, but Modest Individual Strength**

Both of the two strongest Week 5 findings showed statistically significant associations with appointment outcome (p < 0.0001 for both).

However, their individual effect sizes were modest:

- **Booking Lead Time:** Cramér's V = 0.167
- **Previous No-Show History:** Cramér's V = 0.118

Booking lead time showed the stronger association of the two, although both effect sizes indicate relatively modest overall association strength.

This means that while both variables are statistically associated with appointment outcome, neither factor alone is sufficient to explain No-Show behaviour. The findings support considering these variables alongside other characteristics rather than treating either one as a standalone predictor.

**10. The Combined Segment Shows the Strongest Association Among the Tested Factors**

Booking lead time and previous No-Show history were also examined together as a six-category combined variable.

The combined segment produced:

- **Chi-square = 358.52**
- **p < 0.0001**
- **Cramér's V = 0.189**

This was the highest Cramér's V among the individual and combined factors tested:

| Factor | Chi-square | Cramér's V |
|---|---:|---:|
| Booking Lead Time | 280.09 | 0.167 |
| Previous No-Show History | 69.86 | 0.118 |
| **Combined Segment** | **358.52** | **0.189** |

The higher Cramér's V indicates that the combined six-category segmentation has the strongest observed association with appointment outcome among these tested relationships.

This provides additional support for analysing booking lead time and previous No-Show history together rather than examining each characteristic separately.

However, this result does not establish a causal interaction or prove that the two variables independently improve prediction. Further predictive modelling and validation by the Data Science track would be required to determine whether the combined information improves No-Show prediction.

**11. The Lead Time Pattern Remains Consistent Across Multiple Segments**

The booking lead-time relationship was examined across appointment type, distance band, and reminder status.

The escalating No-Show pattern remained visible across all four appointment types, with the 3+ week No-Show rates ranging from **54.4% to 59.9%**. The increase from same-week bookings to 3+ week bookings ranged from **26.1 to 30.5 percentage points**.

The same pattern was also observed across reminder-status groups:

- **No reminder:** 29.9% → 59.6% (**29.7-point increase**)
- **Reminder sent:** 27.1% → 55.9% (**28.8-point increase**)

The pattern was also observed across all three distance bands.

These results provide additional evidence that the Week 5 lead-time finding is not concentrated within a single appointment type, distance group, or reminder-status group.

However, these subgroup analyses are descriptive and do not formally establish the absence of confounding or statistical independence between the variables.

**12. The Highest-Observed Segment Remains Consistent Under Deeper Analysis**

The Week 5 highest-observed segment — **3+ weeks of booking lead time combined with previous No-Show history** — had a No-Show rate of **64.1%** overall.

The segment was further examined across appointment type and reminder status.

Across appointment types, the No-Show rate ranged from **61.9% to 68.0%**:

- Diagnostic Test: **68.0%**
- Follow-up: **64.8%**
- General Consultation: **63.3%**
- Specialist Consultation: **61.9%**

Across reminder status, the same segment showed:

- **No reminder:** 67.4%
- **Reminder sent:** 62.8%

Both reminder groups had substantial sample sizes, with **368** records in the no-reminder group and **966** records in the reminder-sent group.

The consistently high observed No-Show rates across these subgroups provide additional evidence that this segment is not concentrated within a single appointment type or reminder-status group.

The 62.8% rate among appointments where a reminder was sent also remained above the overall dataset No-Show rate of 48.5%. This suggests that reminder status alone does not explain the high observed No-Show rate within this segment.

However, these findings do not establish that reminders are ineffective or that a particular intervention would reduce No-Shows. Further predictive and intervention-focused analysis would be required.

---

## Business Recommendations (Updated Week 6)

The Week 5 recommendations are retained and refined based on the deeper segmentation and statistical validation performed in Week 6. The recommendations focus on the strongest observed patterns while recognising that the analysis is descriptive and does not establish causation.

1. **Prioritise reducing long booking lead times where operationally feasible.**  
   Booking lead time showed the strongest individual association with appointment outcome among the factors formally tested (χ²=280.09, Cramér's V=0.167). The increase in No-Show rates remained consistent across appointment type, distance bands, and reminder-status groups. The clinic could investigate whether more near-term appointment availability or an additional confirmation step for bookings made 3+ weeks in advance would be practical interventions to pilot.

2. **Pilot a targeted intervention for the highest-observed segment.**  
   Appointments combining **3+ weeks of booking lead time with previous No-Show history** had an observed No-Show rate of **64.1%**, while the combined six-category variable showed the highest Cramér's V among the tested factors (**0.189**). The observed rate remained high across both reminder-status groups, including **62.8%** among appointments where a reminder was sent. This suggests that a targeted intervention beyond the standard reminder process could be considered for testing, such as an additional confirmation contact or earlier outreach.

3. **Treat reminders as a supporting measure rather than a standalone solution.**  
   Reminder status and channel showed relatively smaller differences in observed No-Show rates than the main lead-time and previous No-Show findings. The booking lead-time pattern also remained visible within both reminder-status groups. Therefore, reminder strategies should continue to be monitored and improved, but should not be relied upon as the sole approach for addressing the high observed No-Show rate among longer-lead-time appointments and patients with previous No-Shows.

4. **Consider distance-related support for patients living 15km+ from the clinic.**  
   The 15km+ distance group had the highest observed No-Show rate across the distance bands, although distance showed a weaker pattern than the main lead-time finding. Where clinically appropriate, the clinic could investigate options such as telehealth or other access-support measures for patients living farther from the clinic.

5. **Prioritise the combined lead-time and previous No-Show information for future predictive modelling.**  
   The combined six-category variable produced the highest Cramér's V among the tested factors (**0.189**), making the combined information a useful candidate for further investigation by the Data Science track. The Data Science team should assess whether incorporating both characteristics, and potentially their interaction, improves predictive performance when compared with models using the variables individually.

---

## Limitations & Considerations (Updated Week 6)

The Week 5 limitations below remain valid and are retained. Week 6 adds further considerations arising from the statistical validation and deeper segmentation performed during the week.

1. **Synthetic dataset**  
   The HealthConnect appointment dataset is fictional and synthetic. The overall **48.5% No-Show rate** is a characteristic of this dataset and should not be treated as a real-world clinic benchmark.

2. **Descriptive and correlational analysis — statistical significance is not causation**  
   Week 6 confirmed that booking lead time and previous No-Show history are statistically associated with appointment outcome (p < 0.0001 for both) using Chi-square tests of independence. However, statistical significance does not establish causation. Other measured or unmeasured factors may contribute to the observed associations.

3. **Statistical significance does not mean strong predictive performance**  
   The effect sizes for the individual factors were modest (Cramér's V = **0.167** for booking lead time and **0.118** for previous No-Show history). The combined six-category segment had a higher Cramér's V of **0.189**, but this still represents an association rather than evidence of predictive performance. The findings therefore should not be interpreted as showing that any single factor can accurately predict individual No-Show behaviour.

4. **No predictive modelling or performance testing was conducted**  
   This analysis validates associations between variables and appointment outcomes; it does not build, train, or evaluate a predictive model. Terms such as **"highest observed segment"** describe historical patterns within this dataset and should not be interpreted as predictions about an individual patient's future behaviour.

5. **Segment-level analysis can produce greater variation in smaller cells**  
   When segmenting the data across multiple dimensions, some combinations contain fewer records than the larger groups. For example, the Diagnostic Test × 2–3 weeks × Previous No-Show segment contains **58 records** and produced an unexpected lower observed No-Show rate than the corresponding No Previous No-Show group.

   This result should therefore be interpreted cautiously and not treated as evidence that previous No-Show history reduces No-Shows for Diagnostic Tests. Reviewing both the observed rate and underlying record count is important when interpreting highly segmented results.

6. **Subgroup consistency does not formally rule out confounding**  
   Week 6 examined the lead-time pattern across appointment type, distance band, and reminder status. The pattern remained visible across these groups, providing additional descriptive support for the Week 5 finding.

   However, these subgroup analyses do not formally establish that the variables are independent or that confounding is absent. More advanced statistical modelling would be required to assess these relationships while controlling for multiple variables simultaneously.

7. **Combined-segment analysis does not establish a causal interaction**  
   The combined Lead Time × Previous No-Show variable produced the highest Cramér's V among the tested relationships (**0.189**). This indicates a stronger observed association with appointment outcome than either individual factor in this analysis.

   However, the result does not by itself establish a causal interaction effect or prove that combining the two variables improves prediction. Predictive modelling and model comparison would be required to evaluate whether the combined information provides additional predictive value.

8. **Reminder analysis does not establish intervention effectiveness**  
   The lead-time pattern remained consistent across reminder-status groups, and the 3+ weeks + Previous No-Show segment continued to show a high observed No-Show rate among appointments where reminders were sent.

   However, this does not prove that reminders are ineffective or that another intervention would produce better outcomes. Evaluating intervention effectiveness would require an appropriate experimental or quasi-experimental design using real operational data.

9. **Synthetic historical variables require cautious interpretation**  
   Variables such as `previous_appointments` and `previous_no_shows` are synthetic and may not represent fully coherent real-world patient histories. Findings involving previous No-Show behaviour should therefore be treated as patterns within this dataset rather than established patient behaviour.

10. **Missing values**  
    `reminder_channel` contains **1,366 missing values (27.3%)** because these records correspond to appointments where no reminder was sent. This represents structural missingness rather than an unknown reminder channel.

    `distance_to_clinic_km` contains **90 missing values (1.8%)**, while `waiting_time_minutes` contains **60 missing values (1.2%)**. These missing values should be considered when extending the analysis or developing future models.

11. **Small subgroup representation**  
    The **"Prefer not to say"** gender category contains **108 records (2.2%)**. Its observed No-Show rate should therefore be interpreted cautiously and should not be given the same weight as the larger gender groups.

12. **Need for real-world validation and Data Science testing**  
    The findings provide evidence-based areas for further investigation, but they have not yet been validated on real operational data or through predictive modelling.

    The Data Science track should evaluate whether booking lead time, previous No-Show history, and the combined information improve predictive performance when considered alongside other available features. Model performance, feature importance, validation results, and any unexpected findings should be incorporated into the final cross-track assessment.

13. **Recommendations should be treated as hypotheses for testing**  
    The business recommendations in this analysis are based on observed patterns and statistical associations. They should be treated as proposed actions or intervention hypotheses rather than proven solutions.

    Before implementation, recommendations such as targeted outreach, additional confirmation steps, or changes to appointment scheduling should be tested against real-world operational constraints and outcome data.

---

## Cross-Track Integration: Data Science Validation

As part of the Week 6 cross-track integration requirement, I collaborated with members of the **Data Science track** to compare the findings from the Data Analytics analysis with predictive modelling results.

The collaboration involved sharing the key Week 6 findings, particularly:

- Booking lead time
- Previous No-Show history
- The combined Lead Time × Previous No-Show segment
- Distance to clinic
- Reminder variables
- Engineered features relevant to No-Show prediction

The Data Science team provided model performance results, feature rankings, validation findings, and an ablation analysis to assess whether the analytical findings added predictive value.

### Data Science Models

The Data Science team tested:

- Logistic Regression
- Random Forest

In one modelling evaluation, Logistic Regression achieved a higher ROC-AUC (**0.669**) than Random Forest (**0.635**), indicating better discrimination in that evaluation. The Data Science collaborator therefore considered Logistic Regression the better-performing and more interpretable model for that setup.

Another Data Science evaluation compared the Week 5 baseline Logistic Regression with refined Logistic Regression and Random Forest models. In that evaluation, the baseline Logistic Regression achieved a ROC-AUC of **0.68**, while the refined Logistic Regression also achieved **0.68** and Random Forest achieved **0.63**.

Because the modelling results came from different modelling evaluations, the results are reported separately rather than treated as one directly comparable experiment.

### Feature Validation

The Data Science analysis provided additional evidence supporting the importance of several variables identified during the Week 5 and Week 6 Data Analytics analysis.

In the Logistic Regression feature analysis provided by the Data Science track, the following variables had statistically significant coefficients at the 0.05 level:

- `booking_lead_days`
- `previous_no_shows`
- `distance_to_clinic_km`
- `prior_no_show_rate`
- `previous_appointments`
- `appointment_day_Wednesday`

Among these variables, `booking_lead_days` and `previous_no_shows` were particularly relevant to the Week 6 Data Analytics findings.

The results therefore provide cross-track support for continuing to investigate booking lead time and previous No-Show history as important characteristics in No-Show modelling.

### Ablation Study and Feature Refinement

An ablation analysis was also shared by the Data Science track to determine whether engineered features added predictive value beyond their underlying variables.

The findings were:

| Feature | Validation Finding | Decision |
|---|---|---|
| `lead_time_bucket` | Removing it had very little effect on AUC, while removing `booking_lead_days` caused a larger decrease | Treat `booking_lead_days` as the more informative representation; `lead_time_bucket` is redundant for modelling |
| `prior_no_show_rate` | Removing either `prior_no_show_rate` or `previous_no_shows` reduced AUC, with `previous_no_shows` carrying more weight | Retain both for further modelling |
| `is_new_patient` | Removing it or `previous_appointments` changed AUC by less than 0.001 | `is_new_patient` is redundant because it is derived directly from `previous_appointments` |
| `reminder_sent` / `reminder_channel` | The variables contain overlapping information and removing both produced little change in AUC | Drop `reminder_sent` and retain `reminder_channel` |
| `distance_to_clinic_km` | Removing distance slightly improved AUC in the reported ablation test | Retain distance as an analytical finding, but do not prioritise it as an independent modelling feature based on this result |

These findings helped refine the interpretation of the Data Analytics results. In particular, the analysis distinguishes between variables that are useful for descriptive analysis and variables that provide additional predictive information after other features are considered.

### Note on Segment Rate Discrepancy

Fatimah's independent analysis of the combined **Lead Time × Previous No-Show** segment produced a highest observed No-Show rate of **70.5%**, compared with **64.1%** in the Data Analytics analysis.

The difference was attributed to a methodological distinction in the denominator: the Data Analytics analysis included all appointment outcomes, including Cancelled appointments, while Fatimah's calculation excluded Cancelled appointments.

This difference highlights the importance of clearly defining the denominator when calculating and comparing No-Show rates across analyses. Although the reported percentages differ, both analyses identified the same overall pattern: No-Show rates were highest among appointments with **long booking lead times and previous No-Show history**.

### Unexpected Data Science Finding

The Data Science error analysis identified an unexpected pattern: among the No-Show cases examined by the model, false negatives were more likely to have received reminders than the No-Show cases that the model correctly identified.

The reported comparison was:

- Reminder received: **80% of false negatives vs 67% of correctly identified No-Shows**
- SMS reminder: **50% of false negatives vs 32% of correctly identified No-Shows**

This suggests that some No-Show cases may not fit the high-risk profile learned by the model, even when a reminder was sent.

This finding is important because it shows that reminder status alone may not be sufficient for identifying No-Show risk and that the model may miss some cases that appear lower-risk based on the available features.

The finding should be investigated further during subsequent predictive modelling and validation.

### Cross-Track Contribution

**Collaborating Track:** Data Science

**Collaborators:** Donald Nwachukwu and Fatimah Odumuyiwa

**Data Analytics provided:**
- Week 5 No-Show findings
- Booking lead-time analysis
- Previous No-Show analysis
- Combined Lead Time × Previous No-Show segmentation
- Statistical validation results
- KPI refinement
- Key business insights requiring predictive validation

**Data Science provided:**
- Logistic Regression and Random Forest model results
- Model performance metrics
- Feature rankings and Logistic Regression coefficients
- Feature significance results
- Engineered feature validation
- Ablation study results
- Model error-analysis findings

**How the collaboration changed the work:**

The Data Science results strengthened the prioritisation of booking lead time and previous No-Show history while also identifying modelling redundancies among some engineered features. The ablation analysis showed that `booking_lead_days` provides more information than the grouped `lead_time_bucket`, while `is_new_patient` adds little beyond `previous_appointments`. It also showed that `distance_to_clinic_km` did not provide additional predictive value in the reported model setup despite its observed bivariate relationship in the Data Analytics analysis.

The collaboration therefore helped distinguish between variables that show descriptive associations and variables that add predictive value when considered alongside other features.

## Cross-Track Integration Evidence

**1. Track collaborated with:**  
**Data Science Track**

**Collaborators:** Donald Nwachukwu and Fatimah Odumuyiwa

**2. Project dependency:**  
The Data Analytics findings required predictive validation to determine whether the factors identified through descriptive and statistical analysis remained useful when considered alongside other variables in predictive models.

**3. Information/output received:**  
I received Logistic Regression and Random Forest results, model performance metrics, feature rankings and coefficients, feature significance results, engineered feature validation, ablation study results, and model error-analysis findings.

**4. Information/output provided:**  
I provided the Data Science collaborators with my Week 5/6 findings on booking lead time, previous No-Show history, the combined Lead Time × Previous No-Show segment, distance, reminder variables, statistical validation results, KPI refinement, and key business insights.

**5. Integration activity completed:**  
I compared the Data Analytics findings with the Data Science modelling results and reviewed the feature validation and ablation results to determine which analytical findings were also useful from a predictive-modelling perspective.

**6. What changed as a result:**  
The collaboration strengthened the prioritisation of booking lead time and previous No-Show history, supported the combined Lead Time × Previous No-Show segment as a candidate feature for further modelling, and identified redundancies among some engineered features. It also showed that distance had limited additional predictive value in the reported model setup despite its observed bivariate relationship.

**7. Evidence:**  
Evidence of the integration includes the Data Science model results, feature analysis, ablation study, error-analysis findings, exchanged analytical findings, and the documented changes incorporated into this Week 6 notebook, including KPI refinement, feature prioritisation, limitations, and recommendations.